# 04 — ROI2 2D hyphal network review (MIP → Cellpose-SAM → skeleton)
Per-timepoint panels **with their skeleton measurements**, plus the growth curves.

**All PROPOSED — not validated.** The metrics are only meaningful if the skeletons actually trace the hyphae in the panels. Runs on the `canmac (pixi)` kernel; no GPU needed (it reads the QC panels + CSV already produced by `canmac.stages.hyphae2d`).

## 0 — bootstrap + load

In [ ]:
# repo-root bootstrap (OOD kernel CWD is notebooks/, not the repo root)
import sys, os, pathlib
_here = pathlib.Path.cwd()
_root = next((r for r in (_here, *_here.parents) if (r/"canmac"/"__init__.py").exists()),
             pathlib.Path("/vast/scratch/users/kriel.j/monash_lsm"))
sys.path.insert(0, str(_root)); os.chdir(_root)
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
DATASET, QC = "ROI2", pathlib.Path("results/ROI2/qc")
df = pd.read_csv("results/ROI2/hyphae2d.csv").sort_values("t").reset_index(drop=True)
print("cwd", pathlib.Path.cwd(), "| frames:", len(df))

## 1 — full skeleton table

In [ ]:
# Full per-timepoint skeleton table (PROPOSED — confirm against the panels below)
cols = ["t","hours","total_length_um","n_branches","n_tips","n_junctions",
        "longest_branch_um","mean_branch_um","fg_area_um2","n_components"]
display(df[cols].style.format({"hours":"{:.2f}","total_length_um":"{:.1f}",
        "longest_branch_um":"{:.1f}","mean_branch_um":"{:.1f}","fg_area_um2":"{:.0f}"})
        .background_gradient(subset=["total_length_um","fg_area_um2"], cmap="Reds"))

## 2 — growth curves (real DeltaT time axis)

In [ ]:
# Growth curves on the REAL DeltaT time axis (not frame index)
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
ax[0,0].plot(df.hours, df.total_length_um, "o-", ms=3, color="tab:red")
ax[0,0].set_ylabel("total skeleton length (um)"); ax[0,0].set_title("PROPOSED hyphal network growth")
ax[0,1].plot(df.hours, df.n_tips, "o-", ms=3, label="tips")
ax[0,1].plot(df.hours, df.n_junctions, "s-", ms=3, label="junctions")
ax[0,1].set_ylabel("count"); ax[0,1].legend(); ax[0,1].set_title("tips / branch points")
ax[1,0].plot(df.hours, df.fg_area_um2, "o-", ms=3, color="tab:green")
ax[1,0].set_ylabel("mask area (um^2)"); ax[1,0].set_title("foreground area")
ax[1,1].plot(df.hours, df.n_components, "o-", ms=3, color="tab:purple")
ax[1,1].set_ylabel("connected components"); ax[1,1].set_title("fragmentation (high = unstable segmentation)")
for a in ax.ravel(): a.set_xlabel("time (h)"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("NOTE: erratic, non-monotonic curves usually mean frame-to-frame SEGMENTATION instability,")
print("not biology — hyphae do not shrink. Check the per-frame panels below.")

## 3 — per-timepoint: panel + skeleton data side by side

In [ ]:
def review(t, width=1100):
    """Show ONE timepoint: its QC panel (raw MIP | mask | skeleton on raw) + its skeleton metrics."""
    row = df[df.t == t]
    if row.empty:
        print(f"t{t:03d} not in the table"); return
    r = row.iloc[0]
    display(Markdown(
        f"### t{t:03d} — {r.hours:.2f} h &nbsp;|&nbsp; "
        f"**length {r.total_length_um:.1f} um** &nbsp;|&nbsp; tips {int(r.n_tips)} &nbsp;|&nbsp; "
        f"junctions {int(r.n_junctions)} &nbsp;|&nbsp; branches {int(r.n_branches)} &nbsp;|&nbsp; "
        f"longest {r.longest_branch_um:.1f} um &nbsp;|&nbsp; mean {r.mean_branch_um:.1f} um &nbsp;|&nbsp; "
        f"area {r.fg_area_um2:.0f} um² &nbsp;|&nbsp; components {int(r.n_components)}"))
    p = QC / f"hyphae2d_t{t:03d}.png"
    display(Image(filename=str(p), width=width)) if p.exists() else print("no panel:", p)

# EDIT this list to review any timepoints (panel + skeleton data side by side)
for t in [0, 12, 24, 36, 48, 60, 72, 84, 96, 108, 119]:
    review(t)

## 4 — montage (skeleton thumbnails, labelled with length)

In [ ]:
# Thumbnail montage: skeleton-on-raw for every Nth frame, labelled with its length
from matplotlib.image import imread
STEP = 6
ts = [int(t) for t in df.t[::STEP]]
nc = 5; nr = int(np.ceil(len(ts)/nc))
fig, axes = plt.subplots(nr, nc, figsize=(4*nc, 3.4*nr)); axes = np.atleast_1d(axes).ravel()
for a, t in zip(axes, ts):
    p = QC / f"hyphae2d_t{t:03d}.png"
    if p.exists():
        im = imread(str(p))
        a.imshow(im[:, 2*im.shape[1]//3:])          # right-hand third = skeleton-on-raw panel
    r = df[df.t == t].iloc[0]
    a.set_title(f"t{t:03d} {r.hours:.1f}h — {r.total_length_um:.0f}um", fontsize=9)
    a.axis("off")
for a in axes[len(ts):]: a.axis("off")
plt.tight_layout(); plt.show()

## 5 — optional: recompute a frame with different settings

In [ ]:
# OPTIONAL — recompute one frame live to try different settings (cellsam needs a GPU node;
# method="threshold" runs on CPU). Shows the new panel + its metrics.
def recompute(t, method="threshold", **kw):
    from canmac.stages.hyphae2d import load_mip, segment_2d, segment_2d_cellsam, skeleton_metrics
    img = load_mip(DATASET, t)
    mask = (segment_2d_cellsam(img, kw) > 0) if method == "cellsam" else segment_2d(img, **kw)
    met, skel = skeleton_metrics(mask)
    print(f"t{t:03d} [{method}] ->", {k: met[k] for k in
          ("total_length_um","n_tips","n_junctions","n_branches","fg_area_um2","n_components")})
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(img, cmap="gray"); ax[0].set_title(f"raw MIP t{t:03d}")
    ax[1].imshow(mask, cmap="gray"); ax[1].set_title(f"mask ({method})")
    ax[2].imshow(img, cmap="gray"); ys, xs = np.where(skel)
    ax[2].scatter(xs, ys, s=0.4, c="red"); ax[2].set_title("skeleton on raw")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
    return met

# recompute(72, method="threshold", normalize="localstd", norm_mask_percentile=85)